# Spectra-Shield AI: GSTI Data Exploration & Baseline PLSR

This notebook demonstrates:
1. Ingestion of the Global Spectra-Trait Initiative (GSTI v1.0.1) dataset.
2. Spectral visualization of leaf reflectance curves across 400–2400 nm.
3. Comparison of chemometric preprocessing transforms (Raw, SNV, Savitzky-Golay).
4. Baseline Partial Least Squares Regression (PLSR) for $V_{cmax25}$ inversion.
5. Variable Importance in Projection (VIP) explainability analysis.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from src.data.loader import load_gsti_data
from src.preprocessing.spectral import StandardNormalVariate, SavitzkyGolayFilter, PseudoAbsorbance

## 1. Load Standardized Leaf Spectroscopy & Vcmax25 Data

In [ ]:
X, y, meta, wavelengths = load_gsti_data(
    raw_dir='../data/raw',
    target_trait='Vcmax25',
    wavelength_min=400,
    wavelength_max=2400
)
print(f"Loaded spectral matrix X: {X.shape}, Target trait y: {y.shape}")

## 2. Visualize Representative Full-Range Reflectance Curves
Notice key physiological landmarks: Green peak (~550 nm), Chlorophyll red absorption (~670 nm), Red-edge rise (~700–750 nm), NIR cellular scattering plateau (800–1100 nm), and SWIR water absorption troughs (~1400 nm & ~1900 nm).

In [ ]:
plt.figure(figsize=(10, 5))
sample_indices = np.random.choice(len(X), size=10, replace=False)
for idx in sample_indices:
    species_name = meta['Species'].iloc[idx] if 'Species' in meta else f"Sample {idx}"
    plt.plot(wavelengths, X[idx] * 100, alpha=0.7, label=species_name)

plt.title("Representative Leaf Reflectance Spectra (GSTI Database)", fontsize=12)
plt.xlabel("Wavelength (nm)", fontsize=11)
plt.ylabel("Reflectance (%)", fontsize=11)
plt.axvspan(400, 700, color='blue', alpha=0.05, label='Visible (PAR)')
plt.axvspan(700, 1300, color='green', alpha=0.05, label='Near-Infrared (NIR)')
plt.axvspan(1300, 2400, color='orange', alpha=0.05, label='Short-Wave Infrared (SWIR)')
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## 3. Inspect Preprocessing Transformations

In [ ]:
snv = StandardNormalVariate()
sg = SavitzkyGolayFilter(window_length=15, polyorder=2)

X_snv = snv.fit_transform(X[:5])
X_sg = sg.fit_transform(X[:5])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(wavelengths, X[:5].T * 100)
axes[0].set_title("Raw Reflectance (%)")
axes[0].set_xlabel("Wavelength (nm)")

axes[1].plot(wavelengths, X_snv.T)
axes[1].set_title("Standard Normal Variate (SNV)")
axes[1].set_xlabel("Wavelength (nm)")

axes[2].plot(wavelengths, X_sg.T * 100)
axes[2].set_title("Savitzky-Golay Smoothed")
axes[2].set_xlabel("Wavelength (nm)")

plt.tight_layout()
plt.show()

## 4. Load Saved Baseline Model & Variable Importance in Projection (VIP)

In [ ]:
artifact = joblib.load('../models/plsr_vcmax25_baseline.joblib')
vip = np.array(artifact['vip_scores'])
waves = np.array(artifact['wavelengths'])

plt.figure(figsize=(10, 4))
plt.plot(waves, vip, color='darkred', lw=1.5, label='VIP Score')
plt.axhline(1.0, color='black', linestyle='--', label='Significance Threshold (VIP = 1.0)')
plt.title(f"Variable Importance in Projection (VIP) — PLSR {artifact['target_trait']}", fontsize=12)
plt.xlabel("Wavelength (nm)", fontsize=11)
plt.ylabel("VIP Score", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()